# 01 - Data Cleaning, Classification & Basic Analysis
Cleans the raw 38-campaign EDM export, classifies each campaign by content type, and runs the Before/After list-consolidation comparison.

In [ ]:
import pandas as pd

df = pd.read_csv('../data/cleaned/0_Full_Cleaned_Dataset.csv')
df['Date Sent'] = pd.to_datetime(df['Date Sent'], format='%Y-%m-%d')

# Convert percentage strings to numeric values
for col in ['Open Rate', 'CTOR', 'Unsubscribe Rate', 'Bounce Rate']:
    df[col] = df[col].astype(str).str.replace('%','').astype(float)

## Define classification function (must run before it is applied)

In [ ]:
def classify(subject):
    if pd.isna(subject):
        return 'Other Newsletter'
    s = str(subject).lower()
    if 'promotion' in s or 'promote' in s:
        return 'Promotion'
    if 'reminder' in s:
        return 'Reminder'
    if any(k in s for k in ['induction','survey','introduce','introducing','feedback']):
        return 'Informational/Onboarding'
    if 'podcast' in s:
        return 'Podcast+Newsletter'
    if 'post event' in s or 'intro to lia' in s:
        return 'Event-based'
    return 'Other Newsletter'

## Apply classification and check category counts

In [ ]:
df['Campaign_Type'] = df['Email Subject'].fillna('').apply(classify)
print(df['Campaign_Type'].value_counts())

## Basic analysis: Before / after May 2026 list consolidation

In [ ]:
df['Period'] = df['Date Sent'].apply(lambda d: 'Before' if d < pd.Timestamp('2026-05-01') else 'After')

before_after = df[df['Date Sent'] >= pd.Timestamp('2026-01-01')].groupby('Period').agg(
    avg_open=('Open Rate', 'mean'),
    campaign_count=('Email Subject', 'count')
).round(2)

print(before_after)